# 03 — Fusion complète des tables INSEE (2007 / 2013 / 2019)

**Objectif** : dans `INSEE.ipynb`, chaque thème (famille, ménage, diplôme, logement,
population, emploi, nationalité, immigration) est chargé et nettoyé séparément.
Ici on **fusionne tous les thèmes d'une même année en une seule table par commune**,
pour obtenir un jeu de variables socio-démographiques exploitable directement dans
un modèle de prédiction.

**Contrainte mémoire (16 Go de RAM)** : on traite **une année à la fois**, on exporte
immédiatement le résultat en CSV, puis on supprime les DataFrames de cette année
(`del` + `gc.collect()`) avant de passer à la suivante. On ne garde donc jamais les
trois années en mémoire simultanément.

**Sortie** :
- `data/processed/INSEE_Fusion_2007.csv`
- `data/processed/INSEE_Fusion_2013.csv`
- `data/processed/INSEE_Fusion_2019.csv`


## 1. Imports et configuration

In [1]:
import os
import sys
import gc
from functools import reduce

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('../..'))
pd.set_option('display.max_columns', 60)

OUTPUT_DIR = os.path.abspath('../../data/processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Détection automatique de la colonne de code commune

⚠️ **Point à vérifier chez toi** : je ne connais pas le nom exact que
`src/preprocessing.py` donne à la colonne de code commune une fois les tables
"nettoyées" (`CODGEO` est le nom brut standard des tables INSEE `BTX_TD_*`, mais tes
fonctions `load_and_clean_insee_*` l'ont peut-être renommée en `code_insee`,
`Code commune`, etc.).

La fonction ci-dessous essaie une liste de noms courants et te dit lequel elle a
trouvé pour chaque DataFrame — **regarde la sortie de la cellule suivante avant de
continuer**, et complète `CANDIDATE_KEYS` si aucun nom ne correspond.


In [2]:
CANDIDATE_KEYS = [
    "CODGEO", "codgeo", "code_insee", "CODE_INSEE",
    "Code commune", "code_commune", "COM", "insee_com",
]

def find_key_column(df: pd.DataFrame, label: str) -> str:
    for cand in CANDIDATE_KEYS:
        if cand in df.columns:
            return cand
    raise KeyError(
        f"[{label}] Aucune colonne de code commune trouvée parmi {CANDIDATE_KEYS}. "
        f"Colonnes disponibles : {list(df.columns)[:20]} ... "
        f"-> ajoute le bon nom dans CANDIDATE_KEYS."
    )


## 3. Fonction générique de fusion pour une année

On fusionne toutes les tables d'une année en **outer join** sur le code commune
(pour ne perdre aucune commune présente dans au moins un thème), avec un suffixe
par thème pour éviter les collisions de colonnes (ex. `LIBGEO` présent partout).

On force ensuite un `downcast` des types numériques (`float64` → `float32`,
`int64` → `int32` quand c'est possible) pour réduire l'empreinte mémoire avant
export — sur des tables à ~35 000 communes x plusieurs dizaines de variables,
ça peut diviser la taille en mémoire par 2.


In [3]:
def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


def fuse_year(tables: dict[str, pd.DataFrame], year: int) -> pd.DataFrame:
    """Fusionne un dict {nom_theme: df} en une seule table par commune.

    tables : ex. {"famille": df_famille_2007, "menage": df_menage_2007, ...}
    """
    prepared = []
    for name, df in tables.items():
        key = find_key_column(df, name)
        print(f"  - {name:14s}: {df.shape[0]:>6} lignes, {df.shape[1]:>3} colonnes, clé = '{key}'")
        df = df.rename(columns={key: 'code_insee'})
        df['code_insee'] = df['code_insee'].astype(str).str.strip().str.zfill(5)
        # suffixe les colonnes non-clés pour tracer leur origine et éviter les collisions
        rename_map = {c: f"{c}__{name}" for c in df.columns if c != 'code_insee'}
        df = df.rename(columns=rename_map)
        prepared.append(df.drop_duplicates(subset='code_insee'))

    fused = reduce(
        lambda left, right: pd.merge(left, right, on='code_insee', how='outer'),
        prepared,
    )
    fused = downcast_numeric(fused)
    fused['annee_insee'] = year
    print(f"  => table fusionnée {year} : {fused.shape}")
    return fused


## 4. Année 2007

In [4]:
from src.preprocessing import (
    load_and_clean_insee_2007_famille,
    load_and_clean_insee_2007_menage,
    load_and_clean_insee_2007_diplome,
    load_and_clean_insee_2007_logement,
    load_and_clean_insee_2007_population,
    load_and_clean_insee_2007_population2,
    load_and_clean_insee_2007_nationalite,
    load_and_clean_insee_2007_nationalite2,
    load_and_clean_insee_2007_immigration,
    load_and_clean_insee_2007_emploi,
)

tables_2007 = {
    "famille": load_and_clean_insee_2007_famille(sheet="COM"),
    "menage": load_and_clean_insee_2007_menage(sheet="COM"),
    "diplome": load_and_clean_insee_2007_diplome(sheet="COM"),
    "logement": load_and_clean_insee_2007_logement(sheet="COM"),
    "population": load_and_clean_insee_2007_population(sheet="COM"),
    "population2": load_and_clean_insee_2007_population2(sheet="COM"),
    "nationalite": load_and_clean_insee_2007_nationalite(sheet="COM"),
    "nationalite2": load_and_clean_insee_2007_nationalite2(sheet="COM"),
    "immigration": load_and_clean_insee_2007_immigration(sheet="COM"),
    "emploi1": load_and_clean_insee_2007_emploi(sheet="1_COM"),
    "emploi2": load_and_clean_insee_2007_emploi(sheet="2_COM"),
}

print("Tables chargées pour 2007 :")
df_insee_2007 = fuse_year(tables_2007, year=2007)


Tables chargées pour 2007 :
  - famille       : 4401840 lignes,   4 colonnes, clé = 'CODGEO'
  - menage        : 5282208 lignes,   4 colonnes, clé = 'CODGEO'
  - diplome       : 8877044 lignes,   5 colonnes, clé = 'CODGEO'
  - logement      : 4401840 lignes,   5 colonnes, clé = 'CODGEO'
  - population    : 4842024 lignes,   5 colonnes, clé = 'CODGEO'
  - population2   : 4842024 lignes,   5 colonnes, clé = 'CODGEO'
  - nationalite   : 586912 lignes,   5 colonnes, clé = 'CODGEO'
  - nationalite2  : 1173824 lignes,   5 colonnes, clé = 'CODGEO'
  - immigration   : 3521472 lignes,   6 colonnes, clé = 'CODGEO'
  - emploi1       : 9353910 lignes,   5 colonnes, clé = 'CODGEO'
  - emploi2       : 2017510 lignes,   5 colonnes, clé = 'CODGEO'
  => table fusionnée 2007 : (36323, 45)


In [5]:
out_path_2007 = os.path.join(OUTPUT_DIR, "INSEE_Fusion_2007.csv")
df_insee_2007.to_csv(out_path_2007, index=False, sep=";")
print(f"Exporté -> {out_path_2007} ({df_insee_2007.shape})")

# Libère la mémoire avant de passer à 2013
del tables_2007, df_insee_2007
gc.collect()


Exporté -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2007.csv ((36323, 45))


0

## 5. Année 2013

In [6]:
from src.preprocessing import (
    load_and_clean_insee_2013_famille,
    load_and_clean_insee_2013_menage,
    load_and_clean_insee_2013_diplome,
    load_and_clean_insee_2013_logement,
    load_and_clean_insee_2013_population,
    load_and_clean_insee_2013_population2,
    load_and_clean_insee_2013_nationalite,
    load_and_clean_insee_2013_nationalite2,
    load_and_clean_insee_2013_immigration,
    load_and_clean_insee_2013_emploi,
)

tables_2013 = {
    "famille": load_and_clean_insee_2013_famille(sheet="COM"),
    "menage": load_and_clean_insee_2013_menage(sheet="COM"),
    "diplome": load_and_clean_insee_2013_diplome(sheet="COM"),
    "logement": load_and_clean_insee_2013_logement(sheet="COM"),
    "population": load_and_clean_insee_2013_population(sheet="COM"),
    "population2": load_and_clean_insee_2013_population2(sheet="COM"),
    "nationalite": load_and_clean_insee_2013_nationalite(sheet="COM"),
    "nationalite2": load_and_clean_insee_2013_nationalite2(sheet="COM"),
    "immigration": load_and_clean_insee_2013_immigration(sheet="COM"),
    "emploi": load_and_clean_insee_2013_emploi(sheet="COM"),
}

print("Tables chargées pour 2013 :")
df_insee_2013 = fuse_year(tables_2013, year=2013)


Tables chargées pour 2013 :
  - famille       : 4396920 lignes,   5 colonnes, clé = 'CODGEO'
  - menage        : 5276304 lignes,   5 colonnes, clé = 'CODGEO'
  - diplome       : 3224408 lignes,   6 colonnes, clé = 'CODGEO'
  - logement      : 4396920 lignes,   6 colonnes, clé = 'CODGEO'
  - population    : 4836612 lignes,   6 colonnes, clé = 'CODGEO'
  - population2   : 6448816 lignes,   6 colonnes, clé = 'CODGEO'
  - nationalite   : 586256 lignes,   6 colonnes, clé = 'CODGEO'
  - nationalite2  : 1172512 lignes,   6 colonnes, clé = 'CODGEO'
  - immigration   : 2638152 lignes,   7 colonnes, clé = 'CODGEO'
  - emploi        : 8024379 lignes,   6 colonnes, clé = 'CODGEO'
  => table fusionnée 2013 : (36641, 51)


In [7]:
out_path_2013 = os.path.join(OUTPUT_DIR, "INSEE_Fusion_2013.csv")
df_insee_2013.to_csv(out_path_2013, index=False, sep=";")
print(f"Exporté -> {out_path_2013} ({df_insee_2013.shape})")

del tables_2013, df_insee_2013
gc.collect()


Exporté -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2013.csv ((36641, 51))


147592

## 6. Année 2019

In [9]:
from src.preprocessing import (
    load_and_clean_insee_2019_famille,
    load_and_clean_insee_2019_menage,
    load_and_clean_insee_2019_diplome,
    load_and_clean_insee_2019_logement,
    load_and_clean_insee_2019_population,
    load_and_clean_insee_2019_population2,
    load_and_clean_insee_2019_nationalite,
    load_and_clean_insee_2019_nationalite2,
    load_and_clean_insee_2019_immigration,
    load_and_clean_insee_2019_emploi,
)

tables_2019 = {
    "famille": load_and_clean_insee_2019_famille(sheet="COM"),
    "menage": load_and_clean_insee_2019_menage(sheet="COM"),
    "diplome": load_and_clean_insee_2019_diplome(sheet="COM"),
    "logement": load_and_clean_insee_2019_logement(sheet="COM"),
    "population": load_and_clean_insee_2019_population(sheet="COM"),
    "population2": load_and_clean_insee_2019_population2(sheet="COM"),
    "nationalite": load_and_clean_insee_2019_nationalite(sheet="COM"),
    "nationalite2": load_and_clean_insee_2019_nationalite2(sheet="COM"),
    "immigration": load_and_clean_insee_2019_immigration(sheet="COM"),
    "emploi": load_and_clean_insee_2019_emploi(sheet="COM"),
}

print("Tables chargées pour 2019 :")
df_insee_2019 = fuse_year(tables_2019, year=2019)


Tables chargées pour 2019 :
  - famille       : 4192560 lignes,   5 colonnes, clé = 'CODGEO'
  - menage        : 5030986 lignes,   5 colonnes, clé = 'CODGEO'
  - diplome       : 5380452 lignes,   6 colonnes, clé = 'CODGEO'
  - logement      : 4192560 lignes,   6 colonnes, clé = 'CODGEO'
  - population    : 4611816 lignes,   6 colonnes, clé = 'CODGEO'
  - population2   : 6149088 lignes,   6 colonnes, clé = 'CODGEO'
  - nationalite   : 559008 lignes,   6 colonnes, clé = 'CODGEO'
  - nationalite2  : 1118016 lignes,   6 colonnes, clé = 'CODGEO'
  - immigration   : 2515536 lignes,   7 colonnes, clé = 'CODGEO'
  - emploi        : 10132020 lignes,   6 colonnes, clé = 'CODGEO'
  => table fusionnée 2019 : (34938, 51)


In [10]:
out_path_2019 = os.path.join(OUTPUT_DIR, "INSEE_Fusion_2019.csv")
df_insee_2019.to_csv(out_path_2019, index=False, sep=";")
print(f"Exporté -> {out_path_2019} ({df_insee_2019.shape})")

del tables_2019, df_insee_2019
gc.collect()


Exporté -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2019.csv ((34938, 51))


0

## 7. Contrôle rapide

On recharge juste les en-têtes (`nrows=5`) des trois CSV pour vérifier que tout
s'est bien exporté, sans re-solliciter la RAM.


In [11]:
for year in (2007, 2013, 2019):
    path = os.path.join(OUTPUT_DIR, f"INSEE_Fusion_{year}.csv")
    preview = pd.read_csv(path, sep=";", nrows=5)
    print(f"{year} -> {path} : {preview.shape[1]} colonnes")
    display(preview.head())


2007 -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2007.csv : 45 colonnes


,code_insee,CS2_24__famille,NBENFFR__famille,Nombre__famille,CS2_24__menage,NPERC__menage,Nombre__menage,SEXE__diplome,AGEQ65__diplome,DIPL__diplome,Nombre__diplome,TYPLR__logement,CS1_8__logement,STOCD__logement,Nombre__logement,SEXE__population,AGEQ65__population,TACTR__population,Nombre__population,SEXE__population2,AGEQ65__population2,CS1_8__population2,Nombre__population2,SEXE__nationalite,INATC__nationalite,AGE4__nationalite,Nombre__nationalite,SEXE__nationalite2,INATC__nationalite2,CS1_8__nationalite2,Nombre__nationalite2,SEXE__immigration,AGE4__immigration,TACTR__immigration,IMMI__immigration,Nombre__immigration,SEXE__emploi1,CS3_31__emploi1,NA5__emploi1,Nombre__emploi1,SEXE__emploi2,CS3_31__emploi2,NA5__emploi2,Nombre__emploi2,annee_insee
0,10002.0,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Agriculteurs exploitants,1 personne,0,Hommes,15 à 19 ans,Pas de scolarité,0,Maisons,Agriculteurs exploitants,Propriétaire,4,Hommes,15 à 19 ans,Actifs ayant un emploi,7,Hommes,15 à 19 ans,NaN,7,Hommes,Français,Moins de 15 ans,22,Hommes,Français,Agriculteurs exploitants,4,Hommes,Moins de 15 ans,Actifs ayant un emploi,Immigrés,0,Hommes,Agriculteurs sur petite exploitation,"Agriculture, sylviculture et pêche",0,NaN,Policiers et militaires,"Agriculture, sylviculture et pêche",0,2007
1,10003.0,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Agriculteurs exploitants,1 personne,8,Hommes,15 à 19 ans,Pas de scolarité,0,Maisons,Agriculteurs exploitants,Propriétaire,12,Hommes,15 à 19 ans,Actifs ayant un emploi,9,Hommes,15 à 19 ans,NaN,9,Hommes,Français,Moins de 15 ans,220,Hommes,Français,Agriculteurs exploitants,16,Hommes,Moins de 15 ans,Actifs ayant un emploi,Immigrés,0,Hommes,Agriculteurs sur petite exploitation,"Agriculture, sylviculture et pêche",16,NaN,Policiers et militaires,"Agriculture, sylviculture et pêche",0,2007
2,10004.0,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,4,Agriculteurs exploitants,1 personne,4,Hommes,15 à 19 ans,Pas de scolarité,0,Maisons,Agriculteurs exploitants,Propriétaire,12,Hommes,15 à 19 ans,Actifs ayant un emploi,1,Hommes,15 à 19 ans,NaN,1,Hommes,Français,Moins de 15 ans,30,Hommes,Français,Agriculteurs exploitants,12,Hommes,Moins de 15 ans,Actifs ayant un emploi,Immigrés,0,Hommes,Agriculteurs sur petite exploitation,"Agriculture, sylviculture et pêche",8,NaN,Policiers et militaires,"Agriculture, sylviculture et pêche",0,2007
3,10005.0,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,4,Agriculteurs exploitants,1 personne,0,Hommes,15 à 19 ans,Pas de scolarité,0,Maisons,Agriculteurs exploitants,Propriétaire,4,Hommes,15 à 19 ans,Actifs ayant un emploi,2,Hommes,15 à 19 ans,NaN,2,Hommes,Français,Moins de 15 ans,28,Hommes,Français,Agriculteurs exploitants,4,Hommes,Moins de 15 ans,Actifs ayant un emploi,Immigrés,0,Hommes,Agriculteurs sur petite exploitation,"Agriculture, sylviculture et pêche",0,NaN,Policiers et militaires,"Agriculture, sylviculture et pêche",0,2007
4,10006.0,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,8,Agriculteurs exploitants,1 personne,0,Hommes,15 à 19 ans,Pas de scolarité,0,Maisons,Agriculteurs exploitants,Propriétaire,8,Hommes,15 à 19 ans,Actifs ayant un emploi,26,Hommes,15 à 19 ans,NaN,26,Hommes,Français,Moins de 15 ans,294,Hommes,Français,Agriculteurs exploitants,8,Hommes,Moins de 15 ans,Actifs ayant un emploi,Immigrés,0,Hommes,Agriculteurs sur petite exploitation,"Agriculture, sylviculture et pêche",4,NaN,Policiers et militaires,"Agriculture, sylviculture et pêche",0,2007


2013 -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2013.csv : 51 colonnes


,code_insee,LIBGEO__famille,CS2_24__famille,NBENFFR__famille,Nombre__famille,LIBGEO__menage,NPERC__menage,CS2_24__menage,Nombre__menage,LIBGEO__diplome,AGEQ65__diplome,DIPL__diplome,SEXE__diplome,Nombre__diplome,LIBGEO__logement,CS1_8__logement,STOCD__logement,TYPLR__logement,Nombre__logement,LIBGEO__population,SEXE__population,AGEQ65__population,TACTR__population,Nombre__population,LIBGEO__population2,CS1_8__population2,AGEQ65__population2,SEXE__population2,Nombre__population2,LIBGEO__nationalite,AGE4__nationalite,INATC__nationalite,SEXE__nationalite,Nombre__nationalite,LIBGEO__nationalite2,CS1_8__nationalite2,INATC__nationalite2,SEXE__nationalite2,Nombre__nationalite2,LIBGEO__immigration,SEXE__immigration,AGE4_A__immigration,IMMI__immigration,TACTR__immigration,Nombre__immigration,LIBGEO__emploi,CS3_29__emploi,NA5__emploi,SEXE__emploi,Nombre__emploi,annee_insee
0,1001,L'Abergement-Clémenciat,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,L'Abergement-Clémenciat,1 personne,Agriculteurs exploitants,0,L'Abergement-Clémenciat,15 à 19 ans,"Aucun diplôme ou au plus BEPC, brevet des coll...",Hommes,0,L'Abergement-Clémenciat,Agriculteurs exploitants,Propriétaire,Maisons,4,L'Abergement-Clémenciat,Hommes,15 à 19 ans,Actifs ayant un emploi,2,L'Abergement-Clémenciat,Agriculteurs exploitants,15 à 19 ans,Hommes,0,L'Abergement-Clémenciat,Moins de 15 ans,Français,Hommes,88,L'Abergement-Clémenciat,Agriculteurs exploitants,Français,Hommes,8,L'Abergement-Clémenciat,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,1,L'Abergement-Clémenciat,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,8,2013
1,1002,L'Abergement-de-Varey,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,L'Abergement-de-Varey,1 personne,Agriculteurs exploitants,0,L'Abergement-de-Varey,15 à 19 ans,"Aucun diplôme ou au plus BEPC, brevet des coll...",Hommes,0,L'Abergement-de-Varey,Agriculteurs exploitants,Propriétaire,Maisons,0,L'Abergement-de-Varey,Hommes,15 à 19 ans,Actifs ayant un emploi,1,L'Abergement-de-Varey,Agriculteurs exploitants,15 à 19 ans,Hommes,0,L'Abergement-de-Varey,Moins de 15 ans,Français,Hommes,25,L'Abergement-de-Varey,Agriculteurs exploitants,Français,Hommes,0,L'Abergement-de-Varey,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,0,L'Abergement-de-Varey,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,0,2013
2,1004,Ambérieu-en-Bugey,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Ambérieu-en-Bugey,1 personne,Agriculteurs exploitants,0,Ambérieu-en-Bugey,15 à 19 ans,"Aucun diplôme ou au plus BEPC, brevet des coll...",Hommes,51,Ambérieu-en-Bugey,Agriculteurs exploitants,Propriétaire,Maisons,0,Ambérieu-en-Bugey,Hommes,15 à 19 ans,Actifs ayant un emploi,104,Ambérieu-en-Bugey,Agriculteurs exploitants,15 à 19 ans,Hommes,0,Ambérieu-en-Bugey,Moins de 15 ans,Français,Hommes,1326,Ambérieu-en-Bugey,Agriculteurs exploitants,Français,Hommes,0,Ambérieu-en-Bugey,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,27,Ambérieu-en-Bugey,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,4,2013
3,1005,Ambérieux-en-Dombes,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Ambérieux-en-Dombes,1 personne,Agriculteurs exploitants,0,Ambérieux-en-Dombes,15 à 19 ans,"Aucun diplôme ou au plus BEPC, brevet des coll...",Hommes,3,Ambérieux-en-Dombes,Agriculteurs exploitants,Propriétaire,Maisons,5,Ambérieux-en-Dombes,Hommes,15 à 19 ans,Actifs ayant un emploi,11,Ambérieux-en-Dombes,Agriculteurs exploitants,15 à 19 ans,Hommes,0,Ambérieux-en-Dombes,Moins de 15 ans,Français,Hommes,172,Ambérieux-en-Dombes,Agriculteurs exploitants,Français,Hommes,5,Ambérieux-en-Dombes,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,0,Ambérieux-en-Dombes,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,5,2013
4,1006,Ambléon,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Ambléon,1 personne,Agriculteurs exploitants,0,Ambléon,15 à 19 ans,"Aucun diplôme ou au plus BEPC, brevet des coll.

2019 -> /home/Juliendnte/projects/Prediction_Municipale/data/processed/INSEE_Fusion_2019.csv : 51 colonnes


,code_insee,LIBGEO__famille,CS2_24__famille,NBENFFR__famille,Nombre__famille,LIBGEO__menage,NPERC__menage,CS2_24__menage,Nombre__menage,LIBGEO__diplome,AGEQ65__diplome,DIPL__diplome,SEXE__diplome,Nombre__diplome,LIBGEO__logement,CS1_8__logement,STOCD__logement,TYPLR__logement,Nombre__logement,LIBGEO__population,SEXE__population,AGEQ65__population,TACTR__population,Nombre__population,LIBGEO__population2,CS1_8__population2,AGEQ65__population2,SEXE__population2,Nombre__population2,LIBGEO__nationalite,AGE4__nationalite,INATC__nationalite,SEXE__nationalite,Nombre__nationalite,LIBGEO__nationalite2,CS1_8__nationalite2,INATC__nationalite2,SEXE__nationalite2,Nombre__nationalite2,LIBGEO__immigration,SEXE__immigration,AGE4_A__immigration,IMMI__immigration,TACTR__immigration,Nombre__immigration,LIBGEO__emploi,CS3_29__emploi,NA5__emploi,SEXE__emploi,Nombre__emploi,annee_insee
0,1001,L'Abergement-Clémenciat,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,L'Abergement-Clémenciat,1 personne,Agriculteurs exploitants,5,L'Abergement-Clémenciat,15 à 19 ans,Aucun diplôme ou certificat d'études primaires,Hommes,1,L'Abergement-Clémenciat,Agriculteurs exploitants,Propriétaire,Maisons,10,L'Abergement-Clémenciat,Hommes,15 à 19 ans,Actifs ayant un emploi,8,L'Abergement-Clémenciat,Agriculteurs exploitants,15 à 19 ans,Hommes,0,L'Abergement-Clémenciat,Moins de 15 ans,Français,Hommes,82,L'Abergement-Clémenciat,Agriculteurs exploitants,Français,Hommes,10,L'Abergement-Clémenciat,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,0,L'Abergement-Clémenciat,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,10,2019
1,1002,L'Abergement-de-Varey,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,L'Abergement-de-Varey,1 personne,Agriculteurs exploitants,0,L'Abergement-de-Varey,15 à 19 ans,Aucun diplôme ou certificat d'études primaires,Hommes,0,L'Abergement-de-Varey,Agriculteurs exploitants,Propriétaire,Maisons,0,L'Abergement-de-Varey,Hommes,15 à 19 ans,Actifs ayant un emploi,0,L'Abergement-de-Varey,Agriculteurs exploitants,15 à 19 ans,Hommes,0,L'Abergement-de-Varey,Moins de 15 ans,Français,Hommes,28,L'Abergement-de-Varey,Agriculteurs exploitants,Français,Hommes,0,L'Abergement-de-Varey,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,0,L'Abergement-de-Varey,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,0,2019
2,1004,Ambérieu-en-Bugey,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,1,Ambérieu-en-Bugey,1 personne,Agriculteurs exploitants,0,Ambérieu-en-Bugey,15 à 19 ans,Aucun diplôme ou certificat d'études primaires,Hommes,32,Ambérieu-en-Bugey,Agriculteurs exploitants,Propriétaire,Maisons,0,Ambérieu-en-Bugey,Hommes,15 à 19 ans,Actifs ayant un emploi,127,Ambérieu-en-Bugey,Agriculteurs exploitants,15 à 19 ans,Hommes,0,Ambérieu-en-Bugey,Moins de 15 ans,Français,Hommes,1164,Ambérieu-en-Bugey,Agriculteurs exploitants,Français,Hommes,0,Ambérieu-en-Bugey,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,46,Ambérieu-en-Bugey,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,5,2019
3,1005,Ambérieux-en-Dombes,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Ambérieux-en-Dombes,1 personne,Agriculteurs exploitants,0,Ambérieux-en-Dombes,15 à 19 ans,Aucun diplôme ou certificat d'études primaires,Hommes,1,Ambérieux-en-Dombes,Agriculteurs exploitants,Propriétaire,Maisons,0,Ambérieux-en-Dombes,Hommes,15 à 19 ans,Actifs ayant un emploi,11,Ambérieux-en-Dombes,Agriculteurs exploitants,15 à 19 ans,Hommes,0,Ambérieux-en-Dombes,Moins de 15 ans,Français,Hommes,179,Ambérieux-en-Dombes,Agriculteurs exploitants,Français,Hommes,5,Ambérieux-en-Dombes,Hommes,15 à 24 ans,Immigrés,Actifs ayant un emploi,0,Ambérieux-en-Dombes,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,0,2019
4,1006,Ambléon,Agriculteurs exploitants,Aucun enfant de moins de 25 ans,0,Ambléon,1 personne,Agriculteurs exploitants,0,Ambléon,15 à 19 ans,Aucun diplôme ou certificat d'études primaires,Hommes,0,Ambléon,A